# LLM CLeaning Pipeline

## Import Libraries

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.llm_client import call_llm, LLM_PROVIDER
import json
import pandas as pd

BASE_CLEANING_REPORT_OUTPUT_PATH = CLEANING_REPORT_DIR
BASE_LLM_CLEANING_OUTPUT_PATH = CLEANING_REPORT_DIR / "LLM"
BASE_LLM_CLEANING_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

PATH_COMPLETE_AUDIT_LOG = BASE_CLEANING_REPORT_OUTPUT_PATH / "Complete_Cleaning_Audit_Log.csv"


def is_llm_error(text):
    return isinstance(text, str) and text.startswith("[LLM Error]")


print("Libraries imported from utils. LLM client ready.")
print(f"LLM provider: {LLM_PROVIDER.upper()}")

## Data and Log Preparation

In [ ]:
# Load dataset and audit log generated in the previous classical pipeline
try:
    complete_audit_df = pd.read_csv(PATH_COMPLETE_AUDIT_LOG)
    print(f"Complete Audit Log loaded successfully: {len(complete_audit_df)} records found.")

    # Summarize audit log by table and step for LLM consumption
    audit_summary_by_table = {}
    for table_name, group in complete_audit_df.groupby("Table"):
        summary = group.groupby("step").agg(rows_changed=("row_index", "count"), columns_affected=("column", lambda x: ", ".join(x.unique()))).reset_index().to_dict("records")
        audit_summary_by_table[table_name] = summary

except FileNotFoundError:
    print("Complete Audit Log not found. Run Notebook 5 first.")
    audit_summary_by_table = {}

print("Audit Log and Data ready for LLM consumption.")

## Audit Log into Narrative Governance and Non-technical Explaination

In [ ]:
if audit_summary_by_table:
    SYSTEM_NARRATIVE = """You are the Lead Data Governance Officer at an aviation analytics firm.
    Review technical ETL audit logs and translate them into a clear Data Governance Report.
    Focus on data integrity, compliance, and how these cleaning steps improve reliability."""

    for table_name, summary in audit_summary_by_table.items():
        print(f"\nGenerating Governance Report for table: {table_name.upper()}...")

        PROMPT_NARRATIVE = f"""Review the automated data cleaning pipeline executed on the '{table_name}' table.
        
        AUDIT SUMMARY: {json.dumps(summary, indent=2)}
        
        Write a concise Data Governance Report explaining:
        1. What cleaning actions were performed on this specific table.
        2. Why these actions were necessary for business analytics (tailor the reason to what this table holds: e.g., weather, flights, aircraft).
        3. Assurance that raw data was preserved via immutable audit trails."""

        audit_narrative = call_llm(PROMPT_NARRATIVE, system=SYSTEM_NARRATIVE)
        if is_llm_error(audit_narrative):
            print(f"[{table_name}] SKIPPED: governance report failed -> {audit_narrative}")
            continue

        out_file = BASE_LLM_CLEANING_OUTPUT_PATH / f"Governance_Report_{table_name}.md"
        with open(out_file, "w", encoding="utf-8") as f:
            f.write(audit_narrative)

        print(f"Generated: {out_file}")
else:
    print("No audit logs to process for Governance Reports.")

## Categorical Canonicalization

In [ ]:
canonicalization_targets = {
    "Active_Weather": {"file": "ActiveWeather.csv", "col": "weather_description", "hints": "['No Weather', 'Weather', 'Significant Weather']"},
    "Cancellation": {
        "file": "Cancellation.csv",
        "col": "cancellation_reason",
        "hints": "['Not Cancelled', 'Carrier Cancellation', 'Weather Cancellation', 'National Air System Cancellation', 'Security Cancellation']",
    },
    "Aircraft_Range": {"file": "Aircraft.csv", "col": "range", "hints": "['Long Range', 'Medium Range', 'Short Range']"},
    "Aircraft_Width": {"file": "Aircraft.csv", "col": "width", "hints": "['Narrow-body', 'Wide-body']"},
}

In [ ]:
SYSTEM_ENUM = """You are an Aviation Data Steward. Map dirty categorical values into clean, canonical forms.
Return ONLY a valid Python dictionary named `mapping_dict`. No prose outside comments."""

for target_name, config in canonicalization_targets.items():
    path_raw = RECONCILED_DATA_DIR / config["file"]

    try:
        df_raw = pd.read_csv(path_raw)
        col_name = config["col"]

        if col_name in df_raw.columns:
            real_dirty_variants = df_raw[col_name].dropna().unique()[:40].tolist()

            print(f"\n[{target_name}] Extracted {len(real_dirty_variants)} dirty variants for '{col_name}'. Generating mapping...")

            PROMPT_ENUM = f"""Create a mapping dictionary to canonicalize these dirty categorical values from our aviation database.
            Group them into these logical canonical categories: {config['hints']}.
            
            Dirty variants to map: {real_dirty_variants}
            
            Output format MUST be exactly:
            mapping_dict = {{
                'Canonical_Value_1': ['dirty_1', 'dirty_2'],
                'Canonical_Value_2': ['dirty_3']
            }}"""

            enum_code = call_llm(PROMPT_ENUM, system=SYSTEM_ENUM)
            if is_llm_error(enum_code):
                print(f"[{target_name}] SKIPPED: canonicalization map failed -> {enum_code}")
                continue

            out_file = BASE_LLM_CLEANING_OUTPUT_PATH / f"Mapping_{target_name}.py"
            with open(out_file, "w", encoding="utf-8") as f:
                f.write(enum_code.replace("```python", "").replace("```", "").strip())

            print(f"[{target_name}] Mapping saved to: Mapping_{target_name}.py")
        else:
            print(f"[{target_name}] Column '{col_name}' not found in file {config['file']}.")

    except FileNotFoundError:
        print(f"[{target_name}] Original file not found: {path_raw}")

print("\nGeneration of canonicalization maps completed.")

## Architectural Choice Comparisons

In [ ]:
SYSTEM_STRATEGY = """You are a Senior Data Architect designing a modern Data Warehouse for a commercial airline.
Provide objective, structured architectural advice comparing data integration strategies.
Focus on performance, accuracy, and maintainability in an aviation context."""

PROMPT_STRATEGY = """We need to deduplicate our 'STATIONS' (Airports) dimension table. We have historical records with typos in city names and slight variations in airport names.

Compare two data cleaning strategies:
A) Exact String Matching (after basic lowercasing and stripping).
B) Fuzzy Matching using probabilistic record linkage (e.g., Jaro-Winkler distance on names blocked by State).

Provide:
1. The pros and cons of each approach specifically for airport data.
2. The risk of false positives vs false negatives in this context.
3. Your final architectural recommendation for our ETL pipeline."""

print("Generating Strategy Comparison...")
strategy_comparison = call_llm(PROMPT_STRATEGY, system=SYSTEM_STRATEGY)
if is_llm_error(strategy_comparison):
    print(f"Strategy comparison skipped -> {strategy_comparison}")
else:
    with open(BASE_LLM_CLEANING_OUTPUT_PATH / "Dedup_Strategy_Advice.md", "w", encoding="utf-8") as f:
        f.write(strategy_comparison)

print("Strategy comparison generated successfully.")